# Витрина эквайринга по атрибутам MVP первой волны

Зачем нужна тетрадка:
- коллега задает месяц и запускает ячейки сверху вниз;
- на выходе получает готовую витрину в CSV для дашборда.

Ниже — расшифровка полей витрины на русском.

In [ ]:
import pandas as pd

attributes_dict_df = pd.DataFrame([
    {'attribute': 'report_month', 'description_ru': 'Отчетный месяц витрины (первый день месяца)'},
    {'attribute': 'inn', 'description_ru': 'ИНН клиента'},
    {'attribute': 'company_name', 'description_ru': 'Наименование клиента'},
    {'attribute': 'agr_id', 'description_ru': 'ID договора из Alpha (если есть)'},
    {'attribute': 'n_agr', 'description_ru': 'Внутренний номер договора в Alpha'},
    {'attribute': 'contract_number', 'description_ru': 'Номер договора в Alpha'},
    {'attribute': 'd_valid_from', 'description_ru': 'Дата начала действия договора'},
    {'attribute': 'd_valid_to', 'description_ru': 'Дата окончания действия договора'},
    {'attribute': 'cdi_id', 'description_ru': 'Идентификатор клиента в CDI'},
    {'attribute': 'ssp_ocrm', 'description_ru': 'Сегмент/блок ОКРМ'},
    {'attribute': 'ogrn', 'description_ru': 'ОГРН клиента'},
    {'attribute': 'filial_rf', 'description_ru': 'Филиал РФ'},
    {'attribute': 'vsp_name', 'description_ru': 'Наименование ВСП'},
    {'attribute': 'vsp_code', 'description_ru': 'Код ВСП'},
    {'attribute': 'tariff_name', 'description_ru': 'Тарифный план'},
    {'attribute': 'retl_cnt', 'description_ru': 'Количество торговых точек'},
    {'attribute': 'term_cnt', 'description_ru': 'Количество терминалов'},
    {'attribute': 'trx_cnt', 'description_ru': 'Количество операций за месяц'},
    {'attribute': 'trx_sum', 'description_ru': 'Сумма операций за месяц'},
    {'attribute': 'commission_from_ops', 'description_ru': 'Комиссия с операций'},
    {'attribute': 'commission_monthly', 'description_ru': 'Фиксированная комиссия в месяц'},
    {'attribute': 'int_component', 'description_ru': 'Interchange / внутренняя составляющая'},
    {'attribute': 'commission_total', 'description_ru': 'Итоговая комиссия'},
    {'attribute': 'aur', 'description_ru': 'AUR (расчет по терминалам)'},
    {'attribute': 'amortization', 'description_ru': 'Амортизация'},
    {'attribute': 'chod', 'description_ru': 'CHOD'},
    {'attribute': 'fin_result', 'description_ru': 'Финансовый результат'},
])

attributes_dict_df

## 1) Загрузка библиотек

Подключаем только необходимые библиотеки для расчета и выгрузки.

In [ ]:
import re
from decimal import Decimal, InvalidOperation
from pathlib import Path

import pandas as pd
from rail_connectors.connection import connect

## 2) Подключение к Impala

Создаем подключение к хранилищу для SQL-запросов.

In [ ]:
imp = connect(
    to='IMPALA',
    extra_options={'db': 'sandbox_ai'},
    driver_args={'tez.queue.name': 'ai'},
    kerberos={
        'keytab_path': '/home/jovyan/test_requests/tech.keytab',
        'use_credentials': True,
        'update_keytab': True,
    },
    user_params={'user_name': 'Shestopalov-VYur'}
)
imp._init_connection()

## 3) Параметры запуска

Меняем только `report_month` (первый день нужного месяца).
`month_start` и `month_end` считаются автоматически.

In [ ]:
# Единственный параметр месяца (первый день месяца)
report_month = '2026-05-01'

# Путь для итогового CSV
output_csv_path = './datamart_month_acquiring_2026_05.csv'

# Допустимый лаг полной догрузки источников (T-2)
source_lag_days = 2

report_month_ts = pd.to_datetime(report_month)
month_start = report_month_ts.strftime('%Y-%m-%d')
month_end = (report_month_ts + pd.offsets.MonthEnd(1)).strftime('%Y-%m-%d')
report_month_label = report_month_ts.strftime('%Y-%m')

## 4) Сборка витрины по MVP-атрибутам

SQL-центричная сборка с обогащением SA → CDI → CFT → R2 и выбором только согласованных полей.

In [ ]:
# Ячейка оставлена для последовательности; блок проверки готовности источников идет ниже.

### 4.0) Проверка готовности источников (правило T-2)

Перед расчетом проверяем, что обязательные источники (`trx`, `terminals`, SA, OCRM/CDI/CFT, R2) уже загружены и содержат записи за выбранный период.

In [ ]:
current_day = pd.Timestamp.today().normalize()
cutoff_day = current_day - pd.Timedelta(days=source_lag_days)

readiness_checks = [
    {
        'source_name': 'sa_agreement',
        'sql': f"""
            select count(1) as rows_cnt
            from sbx_team_digitcamp.pa_acquiring_agreement a
            where cast(a.d_valid_from as date) <= cast('{month_end}' as date)
              and coalesce(cast(a.d_valid_to as date), cast('2999-12-31' as date)) >= cast('{month_start}' as date)
        """
    },
    {
        'source_name': 'sa_client',
        'sql': """
            select count(1) as rows_cnt
            from sbx_team_digitcamp.pa_acquiring_client
        """
    },
    {
        'source_name': 'trx_month',
        'sql': f"""
            select count(1) as rows_cnt
            from ods.scd1_z_r2_ip_merchants
            where cast(report_month as date) = cast('{month_start}' as date)
        """
    },
    {
        'source_name': 'terminals_ref',
        'sql': """
            select count(1) as rows_cnt
            from ods.scd1_z_r2_ip_merch_dog
        """
    },
    {
        'source_name': 'ocrm_org_ext',
        'sql': """
            select count(1) as rows_cnt
            from ocrm_ul.s_org_ext
            where coalesce(x_removed_flg, 'N') = 'N'
              and coalesce(x_duplicate_flg, 'N') = 'N'
        """
    },
    {
        'source_name': 'cdi_ext_id_org',
        'sql': """
            select count(1) as rows_cnt
            from cdiul.ext_id_org
        """
    },
]

readiness_rows = []
for check in readiness_checks:
    with imp:
        imp.execute('set MEM_LIMIT=2g')
        probe_df = imp.fetch(check['sql'])

    rows_cnt = 0
    if probe_df is not None and len(probe_df):
        rows_cnt = int(pd.to_numeric(probe_df.iloc[0]['rows_cnt'], errors='coerce') or 0)

    readiness_rows.append({
        'source_name': check['source_name'],
        'rows_cnt': rows_cnt,
        'is_ready': rows_cnt > 0,
    })

readiness_df = pd.DataFrame(readiness_rows)
all_sources_ready = bool(readiness_df['is_ready'].all()) if len(readiness_df) else False
lag_rule_ok = report_month_ts <= cutoff_day
run_allowed = all_sources_ready and lag_rule_ok

print(f'Текущая дата: {current_day.date()}')
print(f'Cutoff (T-{source_lag_days}): {cutoff_day.date()}')
print(f'Проверяемый месяц: {report_month_ts.date()}')
print(f'Условие T-{source_lag_days}: {lag_rule_ok}')
print(f'Готовность источников: {all_sources_ready}')

display(readiness_df)

if not lag_rule_ok:
    raise RuntimeError(
        f"Месяц {report_month_label} еще не прошел окно T-{source_lag_days}. "
        f"Запуск разрешен после {cutoff_day.date()}."
    )

if not all_sources_ready:
    bad_sources = readiness_df.loc[~readiness_df['is_ready'], 'source_name'].tolist()
    raise RuntimeError(
        'Не все источники готовы. Проблемные таблицы: ' + ', '.join(bad_sources)
    )

### 4.1) Функции нормализации

Отдельная ячейка с функциями для очистки и приведения данных.

In [ ]:
def normalize_inn(v):
    if pd.isna(v):
        return None
    s = re.sub(r'[^0-9]', '', str(v).strip())
    return s or None


def normalize_contract(v):
    if pd.isna(v):
        return None
    s = str(v).strip()
    return s if s else None


def to_decimal_or_none(v):
    if pd.isna(v):
        return None
    if isinstance(v, Decimal):
        return v
    try:
        return Decimal(str(v).strip().replace(',', '.'))
    except (InvalidOperation, ValueError):
        return None

### 4.2) SA-периметр (базовый слой)

Берем из Озера клиентов и договоры, активные в выбранном месяце, и получаем базовый периметр для последующего обогащения.

In [ ]:
sql_sa_perimeter = f"""
select
  cast(a.n_agr as string) as n_agr,
  cast(a.agr_id as string) as agr_id,
  cast(a.c_contract_num as string) as contract_number,
  cast(a.d_valid_from as date) as d_valid_from,
  cast(a.d_valid_to as date) as d_valid_to,
  regexp_replace(trim(cast(c.inn as string)), '[^0-9]', '') as inn,
  cast(c.name as string) as company_name,
  cast(c.n_cmp as string) as n_cmp,
  cast(c.n_cmp_client as string) as n_cmp_client,
  cast(c.ssp_ocrm as string) as ssp_ocrm,
  cast(c.ogrn as string) as ogrn,
  cast(c.filial_rf as string) as filial_rf,
  cast(c.vsp_name as string) as vsp_name,
  cast(c.vsp_code as string) as vsp_code,
  cast(c.tariff_name as string) as tariff_name,
  cast(c.retl_cnt as bigint) as retl_cnt,
  cast(c.term_cnt as bigint) as term_cnt
from sbx_team_digitcamp.pa_acquiring_agreement a
join sbx_team_digitcamp.pa_acquiring_client c
  on cast(a.n_cmp as string) = cast(c.n_cmp as string)
where cast(a.d_valid_from as date) <= cast('{month_end}' as date)
  and coalesce(cast(a.d_valid_to as date), cast('2999-12-31' as date)) >= cast('{month_start}' as date)
"""

with imp:
    imp.execute('set MEM_LIMIT=8g')
    sa_df = imp.fetch(sql_sa_perimeter)

if sa_df is None:
    sa_df = pd.DataFrame()

if not sa_df.empty:
    sa_df['inn'] = sa_df['inn'].map(normalize_inn)
    sa_df['contract_number'] = sa_df['contract_number'].map(normalize_contract)

sa_rows_cnt = len(sa_df)
sa_inn_unique_cnt = sa_df['inn'].dropna().astype(str).nunique() if 'inn' in sa_df.columns else 0
print(f'Записей в SA-периметре: {sa_rows_cnt:,}')
print(f'Уникальных ИНН в SA-периметре: {sa_inn_unique_cnt:,}')

sa_df.head(5)

### 4.3) Маппинг INN -> CDI

Для ИНН из периметра ищем актуальный OCRM `row_id` и связываем его с `cdi_id` через `cdiul.ext_id_org`.

In [ ]:
inn_values = sorted([x for x in sa_df.get('inn', pd.Series(dtype=object)).dropna().astype(str).unique().tolist() if x])
inn_sql_list = ', '.join([f"'{x}'" for x in inn_values]) if inn_values else "''"

sql_cdi = f"""
with ocrm_current as (
  select
    regexp_replace(trim(cast(se.x_inn as string)), '[^0-9]', '') as inn,
    cast(se.row_id as string) as row_id,
    row_number() over (
      partition by regexp_replace(trim(cast(se.x_inn as string)), '[^0-9]', '')
      order by cast(se.last_upd as timestamp) desc,
               cast(se.created as timestamp) desc,
               cast(se.row_id as string) desc
    ) as rn
  from ocrm_ul.s_org_ext se
  where regexp_replace(trim(cast(se.x_inn as string)), '[^0-9]', '') in ({inn_sql_list})
    and coalesce(se.x_removed_flg, 'N') = 'N'
    and coalesce(se.x_duplicate_flg, 'N') = 'N'
),
ocrm_one as (
  select inn, row_id
  from ocrm_current
  where rn = 1
)
select
  o.inn,
  cast(e.cmo_party_id as string) as cdi_id
from ocrm_one o
left join cdiul.ext_id_org e
  on cast(e.cmo_ext_party_source_id as string) = o.row_id
 and upper(cast(e.cmo_ext_source_system as string)) like 'OCRM%'
"""

with imp:
    imp.execute('set MEM_LIMIT=8g')
    cdi_map_df = imp.fetch(sql_cdi)

if cdi_map_df is None:
    cdi_map_df = pd.DataFrame(columns=['inn', 'cdi_id'])

if not cdi_map_df.empty:
    cdi_map_df['inn'] = cdi_map_df['inn'].map(normalize_inn)
    cdi_map_df['cdi_id'] = cdi_map_df['cdi_id'].astype(str)
    cdi_map_df = cdi_map_df.drop_duplicates(subset=['inn'], keep='first')

cdi_map_df.head(5)

### 4.4) Маппинг CDI -> CFT

Для найденных `cdi_id` получаем соответствующие `cft_id`, чтобы перейти к операционным и финансовым метрикам R2.

In [ ]:
cdi_values = sorted([x for x in cdi_map_df.get('cdi_id', pd.Series(dtype=object)).dropna().astype(str).unique().tolist() if x])
cdi_sql_list = ', '.join([f"'{x}'" for x in cdi_values]) if cdi_values else "''"

sql_cft = f"""
select
  cast(e.cmo_party_id as string) as cdi_id,
  cast(e.cmo_ext_party_source_id as string) as cft_id
from cdiul.ext_id_org e
where cast(e.cmo_party_id as string) in ({cdi_sql_list})
  and upper(cast(e.cmo_ext_source_system as string)) like 'CFT%'
"""

with imp:
    imp.execute('set MEM_LIMIT=8g')
    cft_map_df = imp.fetch(sql_cft)

if cft_map_df is None:
    cft_map_df = pd.DataFrame(columns=['cdi_id', 'cft_id'])

if not cft_map_df.empty:
    cft_map_df['cdi_id'] = cft_map_df['cdi_id'].astype(str)
    cft_map_df['cft_id'] = cft_map_df['cft_id'].astype(str)
    cft_map_df = cft_map_df.drop_duplicates(subset=['cdi_id'], keep='first')

cft_map_df.head(5)

### 4.5) Метрики R2 по CFT

По `cft_id` забираем месячные метрики транзакций и финансового результата из R2 за выбранный отчетный месяц.

In [ ]:
cft_values = sorted([x for x in cft_map_df.get('cft_id', pd.Series(dtype=object)).dropna().astype(str).unique().tolist() if x])
cft_sql_list = ', '.join([f"'{x}'" for x in cft_values]) if cft_values else "''"

sql_r2 = f"""
select
  cast(m.c_cl_org as string) as cft_id,
  cast(m.kod_dog as string) as contract_number_r2,
  cast(m.kol_tst as bigint) as trx_cnt,
  cast(m.sum_tst as decimal(18,2)) as trx_sum,
  cast(m.doh_ekv as decimal(18,2)) as commission_from_ops,
  cast(m.fix_kom as decimal(18,2)) as commission_monthly,
  cast(m.mdr as decimal(18,2)) as int_component,
  cast(m.doh_ekv as decimal(18,2)) + cast(m.fix_kom as decimal(18,2)) as commission_total,
  cast(m.aur as decimal(18,2)) as aur,
  cast(m.amort as decimal(18,2)) as amortization,
  cast(m.nod as decimal(18,2)) as chod,
  cast(m.fin_res as decimal(18,2)) as fin_result
from ods.scd1_z_r2_ip_merchants m
where cast(m.c_cl_org as string) in ({cft_sql_list})
  and cast(m.report_month as date) = cast('{month_start}' as date)
"""

with imp:
    imp.execute('set MEM_LIMIT=8g')
    r2_df = imp.fetch(sql_r2)

if r2_df is None:
    r2_df = pd.DataFrame(columns=[
        'cft_id', 'contract_number_r2', 'trx_cnt', 'trx_sum',
        'commission_from_ops', 'commission_monthly', 'int_component', 'commission_total',
        'aur', 'amortization', 'chod', 'fin_result'
    ])

if not r2_df.empty:
    for c in ['cft_id', 'contract_number_r2']:
        r2_df[c] = r2_df[c].astype(str)

r2_df.head(5)

### 4.6) Финальная сборка MVP-витрины

Объединяем SA, CDI, CFT и R2 в единую таблицу, оставляем только согласованные MVP-атрибуты и готовим итоговый датафрейм к выгрузке.

In [ ]:
base_df = sa_df.copy()

if not cdi_map_df.empty and not base_df.empty:
    base_df = base_df.merge(cdi_map_df[['inn', 'cdi_id']], on='inn', how='left')
else:
    base_df['cdi_id'] = None

if not cft_map_df.empty and not base_df.empty:
    base_df = base_df.merge(cft_map_df[['cdi_id', 'cft_id']], on='cdi_id', how='left')
else:
    base_df['cft_id'] = None

if not r2_df.empty and not base_df.empty:
    base_df = base_df.merge(r2_df, on='cft_id', how='left')
else:
    for col in [
        'contract_number_r2', 'trx_cnt', 'trx_sum',
        'commission_from_ops', 'commission_monthly', 'int_component', 'commission_total',
        'aur', 'amortization', 'chod', 'fin_result'
    ]:
        base_df[col] = None

base_df['report_month'] = report_month_label

mvp_columns = [
    'report_month', 'inn', 'company_name',
    'agr_id', 'n_agr', 'contract_number', 'd_valid_from', 'd_valid_to',
    'cdi_id', 'ssp_ocrm', 'ogrn', 'filial_rf', 'vsp_name', 'vsp_code', 'tariff_name',
    'retl_cnt', 'term_cnt', 'trx_cnt', 'trx_sum',
    'commission_from_ops', 'commission_monthly', 'int_component', 'commission_total',
    'aur', 'amortization', 'chod', 'fin_result'
]

for col in mvp_columns:
    if col not in base_df.columns:
        base_df[col] = None

for col in ['trx_sum', 'commission_from_ops', 'commission_monthly', 'int_component', 'commission_total', 'aur', 'amortization', 'chod', 'fin_result']:
    if col in base_df.columns:
        base_df[col] = base_df[col].map(to_decimal_or_none)

for col in ['retl_cnt', 'term_cnt', 'trx_cnt']:
    if col in base_df.columns:
        base_df[col] = pd.to_numeric(base_df[col], errors='coerce').astype('Int64')

final_df = base_df[mvp_columns].copy()

for forbidden_col in ['n_cmp_client', 'key_tier', 'dq_block_reason']:
    if forbidden_col in final_df.columns:
        final_df = final_df.drop(columns=[forbidden_col])

print(f'Собрано строк: {len(final_df):,}')
print(f'Атрибутов в финале: {len(final_df.columns)}')

## 5) Предпросмотр финальной витрины

Показываем первые 5 строк перед выгрузкой.

In [ ]:
final_df.head(5)

## 6) Выгрузка в CSV

Сохраняем результат в указанный путь.

In [ ]:
output_path = Path(output_csv_path)
output_path.parent.mkdir(parents=True, exist_ok=True)

final_df.to_csv(output_path, index=False, encoding='utf-8-sig')

print(f'CSV сохранен: {output_path.resolve()}')
print(f'Строк: {len(final_df):,}')